# Two-Class Next-Day AQI Model Comparison

This notebook compares XGBoost, Random Forest, Extra Trees, HistGradientBoosting, Logistic Regression, CatBoost, and LightGBM for predicting whether tomorrow is **Clean (AQI ≤ 100)** or **Polluted (AQI > 100)** using the MySQL `air_quality.aqi_daily` table. It includes majority-class and persistence baselines.

The AQI target is rebuilt from all pollutant readings using the shared project rules. Models use today’s six pollutants and nine weather summaries to predict the same station’s next calendar-day class. The database is read-only; the notebook does not save models, CSVs, or database changes.

## Evaluation design

- Split unique dates chronologically: first 80% for training and final 20% for one held-out test period. All station rows for a date stay together.
- Tune parameters only on training dates using four expanding-window time-series folds and macro-F1.
- Compare accuracy, macro/weighted F1, balanced accuracy, and polluted-class precision, recall, F1, and false alerts.
- Add ROC-AUC, average precision (PR-AUC), Brier score, and log loss to assess probability ranking and quality.
- Include majority-class and persistence baselines. Persistence repeats today’s clean/polluted class for tomorrow.

Accuracy can look strong when one class is common. Review polluted-class recall, precision, and false alerts alongside accuracy.

## Why these model families were selected

The data is numeric station-day tabular data. XGBoost is the current boosted-tree reference; Random Forest and Extra Trees provide bagged-tree comparisons; HistGradientBoosting, CatBoost, and LightGBM add other boosting approaches; Logistic Regression is a simpler baseline. This is a focused comparison, not every possible algorithm. SVMs and neural networks could be tested later, but more model types do not automatically solve class imbalance or missing predictive information.

For this task, polluted-day recall must be considered with precision and false alerts.


## Install CatBoost and LightGBM if needed

Run this in a notebook cell using the same kernel, then restart the kernel:

```python
%pip install catboost lightgbm
```

This notebook also requires SQLAlchemy/PyMySQL, scikit-learn, XGBoost, pandas, and NumPy.

## 1. Imports and project settings

The shared AQI helper is located from the repository tree. Set `AIR_QUALITY_DB_URL` (preferred) or `DB_URL` in the environment if your MySQL settings differ from the local default.

In [ ]:
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sqlalchemy import create_engine, text

from sklearn.base import clone
from sklearn.ensemble import (
    ExtraTreesClassifier, HistGradientBoostingClassifier, RandomForestClassifier,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score, brier_score_loss,
    classification_report, confusion_matrix, f1_score, log_loss,
    precision_recall_fscore_support, roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

try:
    from catboost import CatBoostClassifier
    from lightgbm import LGBMClassifier
except ImportError as error:
    raise ImportError(
        "Install the optional models in this notebook kernel with "
        "%pip install catboost lightgbm, then restart the kernel."
    ) from error

HERE = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in [HERE, *HERE.parents]
     if (p / "air-quality-classification" / "app" / "aqi_labels.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from the Data_Science repository or a subfolder.")
APP_DIR = PROJECT_ROOT / "air-quality-classification" / "app"
if str(APP_DIR) not in sys.path:
    sys.path.insert(0, str(APP_DIR))
from aqi_labels import O3_1H_BREAKPOINTS, POLLUTANT_BREAKPOINTS, interpolate_iaqi

DB_URL = os.environ.get(
    "AIR_QUALITY_DB_URL",
    os.environ.get("DB_URL", "mysql+pymysql://root:@localhost/air_quality"),
)
SOURCE_TABLE = "aqi_daily"
POLLUTANTS = ["PM2.5", "PM10", "SO2", "NO2", "CO", "O3"]
WEATHER = [
    "TEMP_mean", "TEMP_range", "PRES_mean", "DEWP_mean", "RH_mean",
    "RAIN_sum", "WSPM_mean", "wd_sin", "wd_cos",
]
FEATURES = POLLUTANTS + WEATHER
REVIEW_FEATURES = ["PM2.5"] + WEATHER
CLASS_NAMES = ["Clean", "Polluted"]
RANDOM_STATE = 42
print("Project:", PROJECT_ROOT)
print("Table:", SOURCE_TABLE, "| full features:", len(FEATURES), "| reviewer:", len(REVIEW_FEATURES))

## 2. Read and validate `aqi_daily`

This cell runs a `SELECT` only, checks the fields required for features and labels, and verifies one row per station and date.

In [ ]:
required = ["date", "station", "station_id", *POLLUTANTS, "O3_1h_max", *WEATHER]
engine = create_engine(DB_URL, pool_pre_ping=True)
try:
    with engine.connect() as connection:
        source = pd.read_sql_query(text(f"SELECT * FROM {SOURCE_TABLE}"), connection)
finally:
    engine.dispose()

missing = [column for column in required if column not in source.columns]
if missing:
    raise KeyError(f"{SOURCE_TABLE} is missing required columns: {missing}")
source["date"] = pd.to_datetime(source["date"], errors="coerce").dt.normalize()
if source.duplicated(["station_id", "date"]).any():
    raise ValueError("aqi_daily contains duplicate station-date keys.")
for column in [*POLLUTANTS, "O3_1h_max", *WEATHER]:
    source[column] = pd.to_numeric(source[column], errors="coerce")

print(f"Rows: {len(source):,}; stations: {source.station.nunique():,}")
print(f"Date range: {source.date.min().date()} through {source.date.max().date()}")
display(source.head())

## 3. Rebuild AQI and exact next-day targets

Official AQI labels are calculated from the full pollutant set. Model features remain today’s pollutant and weather readings. The target join shifts by exactly one calendar day for the same station.

In [ ]:
daily = source.copy()
iaqi_columns = []
for pollutant in POLLUTANTS:
    values = daily[pollutant]
    iaqi_column = f"{pollutant}_iaqi"
    if pollutant == "O3":
        ozone_8h = interpolate_iaqi(values, POLLUTANT_BREAKPOINTS["O3"])
        ozone_1h = interpolate_iaqi(daily["O3_1h_max"], O3_1H_BREAKPOINTS)
        use_one_hour = values.gt(800)
        ozone_8h.loc[use_one_hour] = ozone_1h.loc[use_one_hour]
        daily[iaqi_column] = ozone_8h.round()
    else:
        daily[iaqi_column] = interpolate_iaqi(values, POLLUTANT_BREAKPOINTS[pollutant]).round()
    iaqi_columns.append(iaqi_column)

daily["AQI"] = daily[iaqi_columns].max(axis=1, skipna=True)
today = daily[["date", "station_id", "station", *FEATURES, "AQI"]].copy()
tomorrow = daily[["date", "station_id", "AQI"]].copy()
tomorrow["date"] = tomorrow["date"] - pd.Timedelta(days=1)
tomorrow = tomorrow.rename(columns={"AQI": "target_AQI"})
data = today.merge(
    tomorrow, on=["date", "station_id"], how="inner", validate="one_to_one"
).dropna(subset=["target_AQI"]).copy()
data["target_class"] = (data.target_AQI > 100).astype(int)
data["today_class"] = (data.AQI > 100).astype(float)
data.loc[data.AQI.isna(), "today_class"] = np.nan
data = data.sort_values(["date", "station"]).reset_index(drop=True)

print(f"Exact next-calendar-day examples: {len(data):,}")
class_counts = data.target_class.map({0: "Clean", 1: "Polluted"}).value_counts().reindex(CLASS_NAMES, fill_value=0)
display(class_counts.rename("rows").to_frame())
display(data[["date", "station", "AQI", "target_AQI", "target_class"]].head())

## 4. Chronological train/test split and date-grouped folds

The final 20% of unique dates are held out. All tuning uses only the first 80%.

In [ ]:
unique_dates = np.array(sorted(data.date.dropna().unique()))
cut = int(0.80 * len(unique_dates))
train_dates, test_dates = unique_dates[:cut], unique_dates[cut:]
train = data[data.date.isin(train_dates)].copy().reset_index(drop=True)
test = data[data.date.isin(test_dates)].copy().reset_index(drop=True)

train_date_positions = pd.factorize(train.date, sort=True)[0]
folds = []
for earlier, later in TimeSeriesSplit(n_splits=4).split(np.unique(train_date_positions)):
    folds.append((
        np.flatnonzero(np.isin(train_date_positions, earlier)),
        np.flatnonzero(np.isin(train_date_positions, later)),
    ))

print(f"Training: {len(train):,} rows, {train.date.min().date()} to {train.date.max().date()}")
print(f"Testing: {len(test):,} rows, {test.date.min().date()} to {test.date.max().date()}")
print(f"Four expanding-window folds over {len(train_dates):,} unique training dates.")

## 5. Define classifiers and search spaces

The same model families are evaluated as in the three- and six-class comparisons. All searches use identical training folds and macro-F1 scoring. Most alternatives use class balancing; XGBoost remains the project’s unweighted binary reference. CatBoost disables model-file output.

In [ ]:
XGB_GRID = {
    "classifier__n_estimators": [120, 240],
    "classifier__max_depth": [3, 5],
    "classifier__learning_rate": [0.06],
}
TREE_GRID = {
    "classifier__n_estimators": [300, 600],
    "classifier__max_depth": [16, None],
    "classifier__min_samples_leaf": [3],
}
MODEL_SPECS = {
    "XGBoost": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("classifier", XGBClassifier(
                objective="binary:logistic", eval_metric="logloss",
                random_state=RANDOM_STATE, n_jobs=4, tree_method="hist", verbosity=0,
            )),
        ]), XGB_GRID,
    ),
    "Random Forest": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("classifier", RandomForestClassifier(
                random_state=RANDOM_STATE, n_jobs=-1,
                class_weight="balanced_subsample",
            )),
        ]), TREE_GRID,
    ),
    "Extra Trees": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("classifier", ExtraTreesClassifier(
                random_state=RANDOM_STATE, n_jobs=-1, class_weight="balanced",
            )),
        ]), TREE_GRID,
    ),
    "HistGradientBoosting": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("classifier", HistGradientBoostingClassifier(
                random_state=RANDOM_STATE, class_weight="balanced",
                early_stopping=False,
            )),
        ]), {
            "classifier__max_iter": [100, 200],
            "classifier__max_leaf_nodes": [15, 31],
            "classifier__learning_rate": [0.1],
        },
    ),
    "Logistic Regression": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
            ("classifier", LogisticRegression(
                max_iter=1500, class_weight="balanced", solver="lbfgs",
                random_state=RANDOM_STATE,
            )),
        ]), {"classifier__C": [0.1, 1, 10]},
    ),
    "CatBoost": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("classifier", CatBoostClassifier(
                loss_function="Logloss", auto_class_weights="Balanced",
                random_seed=RANDOM_STATE, thread_count=4, verbose=False,
                allow_writing_files=False,
            )),
        ]), {
            "classifier__iterations": [300, 600],
            "classifier__depth": [4, 6],
            "classifier__learning_rate": [0.05],
        },
    ),
    "LightGBM": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("classifier", LGBMClassifier(
                objective="binary", class_weight="balanced",
                random_state=RANDOM_STATE, n_jobs=4, verbosity=-1,
                force_col_wise=True,
            )),
        ]), {
            "classifier__n_estimators": [200, 400],
            "classifier__num_leaves": [15, 31],
            "classifier__learning_rate": [0.05],
        },
    ),
}
print("Classifiers:", ", ".join(MODEL_SPECS))

## 6. Tune and evaluate each classifier

The held-out period is scored once after selecting each model’s parameters from training folds. Predictions use the model’s highest-probability class.

In [ ]:
def metric_row(name, y_true, prediction, cv_macro_f1=np.nan, best_params=None):
    y_true = np.asarray(y_true).reshape(-1)
    prediction = np.asarray(prediction).reshape(-1)
    p, r, f, support = precision_recall_fscore_support(y_true, prediction, labels=[1], zero_division=0)
    polluted_predictions = int(np.sum(prediction == 1))
    polluted_true_positives = int(np.sum((prediction == 1) & (y_true == 1)))
    return {
        "model": name,
        "cv_macro_f1": cv_macro_f1,
        "accuracy": accuracy_score(y_true, prediction),
        "macro_f1": f1_score(y_true, prediction, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, prediction, average="weighted", zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_true, prediction),
        "polluted_precision": p[0], "polluted_recall": r[0], "polluted_f1": f[0],
        "polluted_support": int(support[0]),
        "polluted_predictions": polluted_predictions,
        "polluted_true_positives": polluted_true_positives,
        "polluted_false_alarms": polluted_predictions - polluted_true_positives,
        "best_params": best_params or "-",
    }

y_test = test.target_class.to_numpy(dtype=int)
searches, fitted_models, predictions, probabilities = {}, {}, {}, {}
result_rows = []
for name, (pipeline, parameter_grid) in MODEL_SPECS.items():
    started = time.time()
    search = GridSearchCV(
        pipeline, parameter_grid, scoring="f1_macro", cv=folds,
        n_jobs=1, refit=True, return_train_score=False, verbose=0,
    )
    search.fit(train[FEATURES], train.target_class)
    best_model = search.best_estimator_
    raw_probability = np.asarray(best_model.predict_proba(test[FEATURES]))
    classes = np.asarray(best_model.named_steps["classifier"].classes_, dtype=int)
    probability = np.zeros((len(test), 2), dtype=float)
    probability[:, classes] = raw_probability
    prediction = probability.argmax(axis=1).astype(int)
    searches[name], fitted_models[name], predictions[name] = search, best_model, prediction
    probabilities[name] = probability
    result_rows.append(metric_row(name, y_test, prediction, search.best_score_, search.best_params_))
    print(
        f"{name}: CV macro-F1={search.best_score_:.4f}; test accuracy={accuracy_score(y_test, prediction):.4f}; "
        f"elapsed={time.time() - started:.1f}s; params={search.best_params_}"
    )

## 7. Add majority and persistence baselines

The majority baseline predicts the most common training class. Persistence repeats today’s Clean/Polluted class; if today’s AQI is unavailable, use the training majority.

In [ ]:
majority_class = int(train.target_class.value_counts().idxmax())
majority_prediction = np.full(len(test), majority_class, dtype=int)
persistence_prediction = test.today_class.fillna(majority_class).to_numpy(dtype=int)
predictions["Majority baseline"] = majority_prediction
predictions["Persistence baseline"] = persistence_prediction
result_rows.extend([
    metric_row("Majority baseline", y_test, majority_prediction),
    metric_row("Persistence baseline", y_test, persistence_prediction),
])
results = pd.DataFrame(result_rows)
RESULT_COLUMNS = [
    "model", "cv_macro_f1", "accuracy", "macro_f1", "weighted_f1",
    "balanced_accuracy", "polluted_precision", "polluted_recall", "polluted_f1",
    "polluted_support", "polluted_true_positives", "polluted_false_alarms",
]
display(results[RESULT_COLUMNS].sort_values("macro_f1", ascending=False).style.format({
    "cv_macro_f1": "{:.4f}", "accuracy": "{:.2%}", "macro_f1": "{:.4f}",
    "weighted_f1": "{:.4f}", "balanced_accuracy": "{:.2%}",
    "polluted_precision": "{:.2%}", "polluted_recall": "{:.2%}", "polluted_f1": "{:.4f}",
}, na_rep="—"))

## 8. Probability ranking and calibration

For each trained classifier, compare ROC-AUC and average precision (PR-AUC) for the Polluted class. Read average precision relative to the polluted-day rate in the test set. Brier score and log loss summarize probability quality; lower is better. These complement, but do not replace, the class-threshold metrics above.

In [ ]:
binary_probability_rows = []
polluted_rate = float(y_test.mean())
for name, probability in probabilities.items():
    polluted_score = probability[:, 1]
    binary_probability_rows.append({
        "model": name,
        "roc_auc": roc_auc_score(y_test, polluted_score),
        "average_precision_pr_auc": average_precision_score(y_test, polluted_score),
        "brier_score": brier_score_loss(y_test, polluted_score),
        "log_loss": log_loss(y_test, probability, labels=[0, 1]),
    })
binary_probability_results = pd.DataFrame(binary_probability_rows)
print(f"Polluted share in test set (no-skill PR-AUC reference): {polluted_rate:.4f}")
display(binary_probability_results.sort_values("average_precision_pr_auc", ascending=False).style.format({
    "roc_auc": "{:.4f}", "average_precision_pr_auc": "{:.4f}",
    "brier_score": "{:.4f}", "log_loss": "{:.4f}",
}))

## 9. Per-class reports and confusion matrices

Rows in each confusion matrix are actual classes; columns are predicted classes.

In [ ]:
for name, prediction in predictions.items():
    print(f"\n{name}")
    report = classification_report(
        y_test, prediction, labels=[0, 1], target_names=CLASS_NAMES,
        output_dict=True, zero_division=0,
    )
    display(pd.DataFrame(report).T.round(4))
    display(pd.DataFrame(
        confusion_matrix(y_test, prediction, labels=[0, 1]),
        index=[f"Actual: {label}" for label in CLASS_NAMES],
        columns=[f"Predicted: {label}" for label in CLASS_NAMES],
    ))

## 10. Final recommendation

The next cell ranks models by accuracy, macro-F1, Polluted-class F1, and PR-AUC. Use the metric that matches the app’s goal: overall classification, balanced performance, or ranking likely polluted days for review.

In [ ]:
model_results = results[~results.model.isin(["Majority baseline", "Persistence baseline"])].copy()
best_accuracy = model_results.loc[model_results.accuracy.idxmax()]
best_macro = model_results.loc[model_results.macro_f1.idxmax()]
best_polluted_f1 = model_results.loc[model_results.polluted_f1.idxmax()]
best_pr_auc = binary_probability_results.loc[binary_probability_results.average_precision_pr_auc.idxmax()]
print(f"Best accuracy: {best_accuracy.model} ({best_accuracy.accuracy:.2%})")
print(f"Best macro-F1: {best_macro.model} ({best_macro.macro_f1:.4f})")
print(
    f"Best Polluted-class F1: {best_polluted_f1.model} ({best_polluted_f1.polluted_f1:.4f}); "
    f"{int(best_polluted_f1.polluted_true_positives)} correct polluted alerts and "
    f"{int(best_polluted_f1.polluted_false_alarms)} false alerts."
)
print(
    f"Best PR-AUC: {best_pr_auc.model} ({best_pr_auc.average_precision_pr_auc:.4f}); "
    f"held-out polluted share is {polluted_rate:.4f}."
)
print("\nPractical reading:")
print("- Prefer macro-F1/balanced accuracy when both classes matter equally.")
print("- Review polluted recall together with precision and false-alert counts before using warnings.")
print("- ROC-AUC/PR-AUC assess ranking across thresholds; they do not select an operational threshold.")
print("- Confirm the chosen model beats majority and persistence baselines on the same holdout.")
print("- Treat this historical holdout as evidence, not a guarantee for future dates.")

## Results summary

When run, this notebook computes a fresh two-class comparison from the current `aqi_daily` table and reports model leaders by accuracy, macro-F1, Polluted-class F1, and PR-AUC. The recommendation displays false-alert counts beside the selected results.

There is no universal winner: if the app warns users, choose a model and probability threshold based on the acceptable balance between missed polluted days and false alerts, then validate on a later untouched period.

The candidates were selected for numeric tabular station-day data. Tree ensembles and boosting methods can model nonlinear patterns and interactions; Logistic Regression provides a simple linear baseline. SVMs and neural networks were not included in this focused benchmark, and testing more algorithms does not by itself solve missing or weak predictive information.